# Cisco Skill Scanner — combined pilot comparison

**MSR 2027 · SkillGuard · RQ1**

`01_cisco_scanner_dataset_ready.ipynb` produces one run folder per experiment. This notebook **combines several
finished run folders** (here: the three pilots) into one comparison report. It does not call Cisco, does not read
the GitSkills database, and does not execute any skill code: it only reads the CSV files the runs exported
(`scan_summary.csv`, `individual_findings.csv`, `context_comparison.csv`, `analyzer_comparison.csv`,
`manual_review_template.csv`) plus each run's `run_metadata.json`.

**Primary vs sensitivity analysis.** As in notebook 01, *primary* results use `COMPLETE` scans only. The
*sensitivity analysis* adds scans whose only problem is Cisco's known rule-registration error (`CONTRACT_ONLY`).
Contract-only scans are never treated as complete or accurate; the sensitivity analysis only shows whether
including them would change a result. `PARTIAL` scans would appear only as preliminary evidence.

**Cisco findings are not confirmed vulnerabilities.** A finding means a Cisco rule matched an operation in the
skill's files. Whether that operation is legitimate, risky or malicious needs manual review.

**Outputs** go to a new folder `results/cisco_dataset_runs/<database>/combined/<date>_<time>_<COMBINED_NAME>/`;
an existing folder is never reused:

| file | contents |
|---|---|
| `combined_report.md` | the written comparison with all summary tables |
| `combined_overview.csv` | one row per pilot: folder, scans by outcome, findings |
| `combined_scan_summary.csv` | every scan of every pilot (`scan_summary.csv` rows + `pilot`) |
| `combined_findings.csv` | every Cisco finding of every pilot (`individual_findings.csv` rows + `pilot`) |
| `combined_pairs.csv` | every comparison pair with its `pair_status` (primary / sensitivity / preliminary / not comparable) |
| `combined_comparison_findings.csv` | finding-level comparison rows of every pilot |
| `combined_manual_review.csv` | distinct security finding locations with the reasons they are worth reviewing, blank reviewer columns |
| `combined_metadata.json` | which run folders were combined, their scanner version, policy fingerprint and file hashes |
| `figures/*.png` | the charts |

## 1 — Settings

`RUNS` maps a short pilot label to a finished run folder (relative to `OUTPUT_ROOT`, or absolute). Every run must
come from the same scanner version, scan policy and set of configurations; Section 3 stops with an error otherwise.

In [ ]:
RUNS = {
    "canary-tokens":   "agent_skills_sample/runs/2026-11-03_152815_pilot-canary-reanalysis",
    "x":               "agent_skills_sample/runs/2026-11-03_153247_pilot-x",
    "loa-credentials": "agent_skills_sample/runs/2026-11-03_153301_pilot-loa-credentials",
}
COMBINED_NAME = "pilots-canary-x-loa"         # label at the end of the output folder name
OUTPUT_ROOT = "results/cisco_dataset_runs"
TOP_REVIEW_EXAMPLES = 15                      # rows shown in the "examples for manual review" table

## 2 — Imports, paths and the output folder

In [ ]:
import hashlib
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

cwd = Path.cwd().resolve()
ROOT = next(f for f in (cwd, *cwd.parents) if (f / "notebooks").is_dir() and (f / "data").is_dir())
OUTPUT_ROOT_PATH = (ROOT / OUTPUT_ROOT).resolve()
RUN_DIRS = {label: (OUTPUT_ROOT_PATH / path).resolve() for label, path in RUNS.items()}
for label, path in RUN_DIRS.items():
    if not (path / "scan_summary.csv").is_file():
        raise FileNotFoundError(f"{label}: no scan_summary.csv in {path}")

db_tags = {json.loads((path / "run_metadata.json").read_text())["database"]["tag"] for path in RUN_DIRS.values()}
if len(db_tags) != 1:
    raise ValueError(f"The runs come from different databases: {sorted(db_tags)}")
OUT_DIR = OUTPUT_ROOT_PATH / db_tags.pop() / "combined" / f"{datetime.now():%Y-%m-%d_%H%M%S}_{COMBINED_NAME}"
OUT_DIR.mkdir(parents=True, exist_ok=False)        # never reuse an existing combined report folder
(OUT_DIR / "figures").mkdir()
print("Combining:", ", ".join(RUN_DIRS))
print("Output   :", OUT_DIR.relative_to(ROOT))

## 3 — Load the runs and check that they are comparable

Each run's tables get a `pilot` column. The checks below stop the notebook if the runs were produced with different
scanner versions, policies or configurations, or if a run's findings table does not match its scan summary
(the number of finding rows must equal the findings Cisco reported in the readable scans).

In [ ]:
SEVERITY_ORDER = ["CRITICAL", "HIGH", "MEDIUM", "LOW", "INFO"]
OUTCOME_ORDER = ["complete", "contract_only", "partial", "failed", "skipped", "missing"]
CHANGE_ORDER = ["unchanged", "changed", "possibly_equivalent", "additional", "missing"]
READABLE = ["complete", "contract_only", "partial"]


def load(label, name):
    path = RUN_DIRS[label] / name
    df = pd.read_csv(path) if path.is_file() else pd.DataFrame()
    return df.assign(pilot=label) if not df.empty else df


scans = pd.concat([load(p, "scan_summary.csv") for p in RUNS], ignore_index=True)
findings = pd.concat([load(p, "individual_findings.csv") for p in RUNS], ignore_index=True)
diffs = pd.concat([load(p, "context_comparison.csv") for p in RUNS] + [load(p, "analyzer_comparison.csv") for p in RUNS], ignore_index=True)
review = pd.concat([load(p, "manual_review_template.csv") for p in RUNS], ignore_index=True)
metadata = {p: json.loads((RUN_DIRS[p] / "run_metadata.json").read_text()) for p in RUNS}

problems = []
readable = scans[scans["outcome"].isin(READABLE)]
if readable["scanner_version"].nunique() != 1:
    problems.append(f"different scanner versions: {sorted(readable['scanner_version'].dropna().unique())}")
if readable["policy_fingerprint"].nunique() != 1:
    problems.append(f"different policy fingerprints: {sorted(readable['policy_fingerprint'].dropna().unique())}")
config_sets = scans.groupby("pilot")["configuration"].apply(frozenset)
if config_sets.nunique() != 1:
    problems.append(f"different configuration sets: {config_sets.to_dict()}")
for p in RUNS:
    if metadata[p].get("stage") != "complete":
        problems.append(f"{p}: run_metadata.json stage is {metadata[p].get('stage')!r}, not 'complete'")
    expected = int(scans.loc[(scans["pilot"] == p) & scans["outcome"].isin(READABLE), "findings_count"].sum())
    got = int((findings["pilot"] == p).sum()) if not findings.empty else 0
    if expected != got:
        problems.append(f"{p}: {got} finding rows but the scan summary reports {expected}")
if problems:
    raise ValueError("Runs are not comparable:\n- " + "\n- ".join(problems))

SCANNER_VERSION = readable["scanner_version"].iloc[0]
POLICY = f"{readable['policy_name'].iloc[0]} / {readable['policy_preset'].iloc[0]} (fingerprint {readable['policy_fingerprint'].iloc[0][:12]}...)"
CONFIG_ORDER = [c for c in ["SKILL.md only | default", "SKILL.md only | behavioral", "Full folder | default",
                            "Full folder | behavioral", "SKILL.md only | llm", "Full folder | llm"] if c in set(scans["configuration"])]
findings["severity"] = pd.Categorical(findings["severity"], categories=SEVERITY_ORDER, ordered=True)
print(f"OK: {len(RUNS)} runs, {len(scans)} scans, {len(findings)} finding records; scanner {SCANNER_VERSION}; policy {POLICY}")
display(scans.pivot_table(index="pilot", columns="outcome", values="configuration", aggfunc="count", fill_value=0)
        .reindex(index=list(RUNS), columns=[o for o in OUTCOME_ORDER if o in set(scans["outcome"])], fill_value=0))

## 4 — Comparison pairs: primary vs sensitivity

The pair rules are the same as in notebook 01 (Section 12): `eligible` = both scans `COMPLETE` (and a complete folder
for SKILL.md-only vs full-folder); `sensitivity: contract-only scan` = at least one `CONTRACT_ONLY`, none partial;
`preliminary` = a partial scan or an incomplete folder; `not comparable` = a failed, skipped or missing scan. The
pairs are rebuilt from the scan summaries so that pairs with **no findings at all** are counted in the sample sizes,
and the result is cross-checked against the `pair_status` saved in each run's comparison CSVs.

In [ ]:
def pair_status(before, after, needs_complete_folder):
    outcomes = {before["outcome"], after["outcome"]}
    if not outcomes <= set(READABLE):
        return f"not comparable ({before['status']} vs {after['status']})"
    if needs_complete_folder and after["folder_complete"] is not True:
        return "preliminary: incomplete folder"
    if "partial" in outcomes:
        return "preliminary: partial scan"
    if "contract_only" in outcomes:
        return "sensitivity: contract-only scan"
    return "eligible"


PRIMARY = ["eligible"]
SENSITIVITY = ["eligible", "sensitivity: contract-only scan"]
modes = list(dict.fromkeys(scans["analyzer_mode"].dropna()))
PAIR_DEFS = ([(f"SKILL.md only -> Full folder: {m}", f"SKILL.md only | {m}", f"Full folder | {m}", True) for m in modes]
             + [(f"default -> {m}: {s}", f"{s} | default", f"{s} | {m}", False) for m in modes if m != "default"
                for s in ("SKILL.md only", "Full folder")])

lookup = {(r["pilot"], r["skill_id"], r["configuration"]): r for r in scans.to_dict("records")}
pair_rows = []
for (pilot, skill_id) in scans[["pilot", "skill_id"]].drop_duplicates().itertuples(index=False):
    for label, before_cfg, after_cfg, needs_folder in PAIR_DEFS:
        before, after = lookup.get((pilot, skill_id, before_cfg)), lookup.get((pilot, skill_id, after_cfg))
        if before and after:
            pair_rows.append({"pilot": pilot, "skill_id": skill_id, "comparison": label,
                              "before": f"{before_cfg} ({before['status']})", "after": f"{after_cfg} ({after['status']})",
                              "pair_status": pair_status(before, after, needs_folder)})
pairs = pd.DataFrame(pair_rows)

# Cross-check with the pair_status each run saved next to its comparison findings.
saved = diffs[["pilot", "skill_id", "comparison", "pair_status"]].drop_duplicates()
check = saved.merge(pairs, on=["pilot", "skill_id", "comparison"], suffixes=("_saved", "_rebuilt"), how="left")
mismatch = check[check["pair_status_saved"] != check["pair_status_rebuilt"]]
if not mismatch.empty:
    raise ValueError(f"Pair status differs from the saved runs:\n{mismatch}")
print(f"{len(pairs)} comparison pairs rebuilt; saved pair statuses agree ({len(saved)} pair(s) with findings checked).")
display(pairs.pivot_table(index="comparison", columns="pilot", values="pair_status", aggfunc="first")
        .reindex(index=[d[0] for d in PAIR_DEFS], columns=list(RUNS)))

## 5 — Summary tables

Every table states its **unit**. "Finding records" count one row per finding per configuration, so a finding that
appears in all four configurations counts four times. "Distinct locations" count each (skill, rule, file, line)
once per pilot, whatever the number of configurations that reported it.

In [ ]:
security = findings[findings["finding_kind"] == "security"]
primary_sec = security[security["run_outcome"] == "complete"]
sensitivity_sec = security[security["run_outcome"].isin(["complete", "contract_only"])]


def distinct_locations(df):
    """One row per (pilot, skill, location) with the highest severity any configuration reported."""
    return df.sort_values("severity").drop_duplicates(["pilot", "skill_id", "location_key"])


def by_pilot(df, column, order=None):
    if df.empty:
        return pd.DataFrame(0, index=list(RUNS), columns=order or [])
    t = df.pivot_table(index="pilot", columns=column, values="finding_key", aggfunc="count", fill_value=0, observed=True)
    t.columns = t.columns.astype(str)
    return t.reindex(index=list(RUNS), columns=order or list(t.columns), fill_value=0).astype(int)


# T1 overview per pilot
overview_rows = []
for p in RUNS:
    s, f = scans[scans["pilot"] == p], findings[findings["pilot"] == p]
    manifest_skill = metadata[p]["skills"][0] if metadata[p].get("skills") else {}
    row = {"pilot": p, "repo / path": f"{s['repo'].iloc[0]} / {s['skill_path'].iloc[0]}",
           "supporting files": manifest_skill.get("supporting_files_exported"), "folder complete": manifest_skill.get("folder_complete")}
    row.update({f"scans {o}": int((s["outcome"] == o).sum()) for o in OUTCOME_ORDER if (scans["outcome"] == o).any()})
    row.update({"finding records (all readable scans)": len(f),
                "security records, primary": int(((f["finding_kind"] == "security") & (f["run_outcome"] == "complete")).sum()),
                "security records, contract-only scans": int(((f["finding_kind"] == "security") & (f["run_outcome"] == "contract_only")).sum()),
                "metadata/coverage records": int((f["finding_kind"] != "security").sum()),
                "distinct security locations, primary": len(distinct_locations(primary_sec[primary_sec["pilot"] == p])),
                "distinct security locations, sensitivity": len(distinct_locations(sensitivity_sec[sensitivity_sec["pilot"] == p]))})
    overview_rows.append(row)
overview = pd.DataFrame(overview_rows)

# T2 findings per pilot and configuration (all finding kinds), with the scan status
cell = lambda r: f"{int(r['findings_count'])} ({r['status']})" if pd.notna(r["findings_count"]) else r["status"]
per_config = scans.assign(cell=scans.apply(cell, axis=1)).pivot_table(index="pilot", columns="configuration", values="cell", aggfunc="first") \
    .reindex(index=list(RUNS), columns=CONFIG_ORDER)
max_sev = scans.pivot_table(index="pilot", columns="configuration", values="max_severity", aggfunc="first").reindex(index=list(RUNS), columns=CONFIG_ORDER)

# T3 severity distributions
sev_records_primary = by_pilot(primary_sec, "severity", SEVERITY_ORDER)
sev_distinct_primary = by_pilot(distinct_locations(primary_sec), "severity", SEVERITY_ORDER)
sev_distinct_sensitivity = by_pilot(distinct_locations(sensitivity_sec), "severity", SEVERITY_ORDER)

# T6 categories and rules (distinct locations, primary; sensitivity adds contract-only scans)
cat_primary = by_pilot(distinct_locations(primary_sec), "cisco_category")
research_primary = by_pilot(distinct_locations(primary_sec), "research_category")
RESEARCH_ORDER = ["Credential / secret access or exposure", "External network communication / data exfiltration",
                  "Command execution / dangerous shell operations", "File-system access or modification",
                  "Prompt injection / instruction manipulation", "Excessive permissions / privilege escalation",
                  "Supply-chain / dependency risks", "Account modification / sensitive external actions"]
research_primary = research_primary.T.reindex(RESEARCH_ORDER + [c for c in research_primary.columns if c not in RESEARCH_ORDER], fill_value=0)
research_primary["pilots_with_findings"] = (research_primary > 0).sum(axis=1)


def rule_table(df):
    if df.empty:
        return pd.DataFrame()
    d = distinct_locations(df)
    return (d.groupby(["rule_id", "cisco_category"]).agg(distinct_locations=("location_key", "size"),
                                                          pilots=("pilot", lambda s: ", ".join(sorted(set(s)))),
                                                          severities=("severity", lambda s: ", ".join(sorted(set(map(str, s))))))
            .sort_values("distinct_locations", ascending=False).reset_index())


rules_primary = rule_table(primary_sec)
rules_contract_only = rule_table(security[(security["run_outcome"] == "contract_only")
                                          & ~security["rule_id"].isin(set(primary_sec["rule_id"]))])
print("T1 overview"); display(overview)
print("\nT2 findings per configuration (all kinds) and scan status"); display(per_config)
print("\nT3 severity of security findings: distinct locations, primary"); display(sev_distinct_primary)
print("\nT6 Cisco categories (distinct security locations, primary)"); display(cat_primary)
print("\nT7 rules (distinct security locations, primary)"); display(rules_primary)

In [ ]:
def comparison_totals(statuses):
    """Per comparison across all pilots: pairs used, pilots used, finding changes summed, pilots gaining findings."""
    rows = []
    for label, *_ in PAIR_DEFS:
        used = pairs[(pairs["comparison"] == label) & pairs["pair_status"].isin(statuses)]
        d = diffs[(diffs["comparison"] == label) & diffs["pair_status"].isin(statuses)] if not diffs.empty else diffs
        row = {"comparison": label, "pairs_used": len(used), "pilots_used": ", ".join(used["pilot"]) or "none"}
        row.update({c: int((d["change"] == c).sum()) if not d.empty else 0 for c in CHANGE_ORDER})
        row["pilots_gaining_findings"] = ", ".join(sorted(set(d.loc[d["change"] == "additional", "pilot"]))) if not d.empty else ""
        rows.append(row)
    return pd.DataFrame(rows)


def per_pilot_changes(statuses):
    if diffs.empty:
        return pd.DataFrame()
    d = diffs[diffs["pair_status"].isin(statuses)]
    t = d.pivot_table(index=["comparison", "pilot"], columns="change", values="finding_key", aggfunc="count", fill_value=0)
    used = pairs[pairs["pair_status"].isin(statuses)].set_index(["comparison", "pilot"]).index
    return t.reindex(index=used, columns=CHANGE_ORDER, fill_value=0).astype(int)


def sample_sizes():
    group = pairs["pair_status"].map(lambda s: "complete" if s == "eligible" else "contract_only" if s.startswith("sensitivity")
                                     else "partial" if "partial" in s else "incomplete_folder" if "incomplete" in s else "not_comparable")
    t = pairs.assign(group=group).pivot_table(index="comparison", columns="group", values="pilot", aggfunc="count", fill_value=0)
    t = t.reindex(index=[d[0] for d in PAIR_DEFS], columns=["complete", "contract_only", "partial", "incomplete_folder", "not_comparable"], fill_value=0)
    return pd.DataFrame({"pairs_total": t.sum(axis=1), "primary_n": t["complete"], "sensitivity_n": t["complete"] + t["contract_only"],
                         "excluded_contract_only": t["contract_only"], "excluded_partial": t["partial"],
                         "excluded_incomplete_folder": t["incomplete_folder"], "excluded_not_comparable": t["not_comparable"]})


totals_primary, totals_sensitivity = comparison_totals(PRIMARY), comparison_totals(SENSITIVITY)
effect_lines = []
for (_, p), (_, s) in zip(totals_primary.iterrows(), totals_sensitivity.iterrows()):
    if p["pairs_used"] == s["pairs_used"]:
        effect_lines.append(f"{p['comparison']}: no contract-only pairs; sensitivity = primary (n = {p['pairs_used']}).")
    else:
        effect_lines.append(f"{p['comparison']}: primary n = {p['pairs_used']} ({p['pilots_used']}), sensitivity n = {s['pairs_used']} "
                            f"({s['pilots_used']}). Additional findings {p['additional']} vs {s['additional']}; unchanged "
                            f"{p['unchanged']} vs {s['unchanged']}; missing {p['missing']} vs {s['missing']}; pilots gaining findings: "
                            f"{p['pilots_gaining_findings'] or 'none'} vs {s['pilots_gaining_findings'] or 'none'}.")
contract_scans = scans[scans["outcome"] == "contract_only"][["pilot", "configuration", "findings_count", "max_severity", "status_detail"]]
invalid = findings[(findings["run_outcome"] == "contract_only") & (findings["cisco_rule_contract"] == "invalid")]

where_new = diffs[(diffs["pair_status"].isin(PRIMARY)) & (diffs["change"] == "additional") & diffs["comparison"].str.startswith("SKILL.md only")] \
    if not diffs.empty else diffs
files_new = where_new.groupby(["comparison", "pilot", "location", "file_path"], dropna=False).agg(
    findings=("finding_key", "size"), rules=("rule_id", lambda s: ", ".join(sorted(set(s))))).reset_index() if not where_new.empty else pd.DataFrame()

print("T4 sample sizes"); display(sample_sizes())
print("\nT5 primary totals"); display(totals_primary)
print("\nT5s sensitivity totals"); display(totals_sensitivity)
for line in effect_lines:
    print("  -", line)

In [ ]:
# Examples that deserve manual review: chosen by explicit rules, never by a judgement about the finding itself.
additional_in_skill_md = set(zip(where_new.loc[where_new["location"] == "SKILL.md", "pilot"],
                                 where_new.loc[where_new["location"] == "SKILL.md", "location_key"])) if not where_new.empty else set()
behavioral_only = set()
if not diffs.empty:
    b = diffs[diffs["comparison"].str.startswith("default ->") & (diffs["change"] == "additional") & diffs["pair_status"].isin(SENSITIVITY)]
    behavioral_only = set(zip(b["pilot"], b["location_key"]))
def place(row):
    """Text key for (pilot, skill, rule, file, line); missing values become '' so they compare equal."""
    return "|".join("" if pd.isna(row[c]) else str(row[c]) for c in ("pilot", "skill_id", "rule_id", "file_path", "reported_line_number"))


def review_reasons(row):
    key = (row["pilot"], location_of.get(place(row)))
    reasons = []
    if any(s in str(row["severity_reported"]) for s in ("CRITICAL", "HIGH")):
        reasons.append("CRITICAL/HIGH severity")
    if key in additional_in_skill_md:
        reasons.append("appears in SKILL.md only when the scripts are present (context re-interpretation)")
    if key in behavioral_only:
        reasons.append("only reported by the behavioral analyzer")
    if row["only_in_non_complete_scans"]:
        reasons.append("seen only in a CONTRACT_ONLY/PARTIAL scan")
    if row["evidence_status"] in ("mismatch", "ambiguous"):
        reasons.append(f"reported line does not hold the evidence ({row['evidence_status']})")
    if row["cisco_category"] == "unauthorized_tool_use":
        reasons.append("manifest vs code capability mismatch")
    return reasons


review["reported_line_number"] = pd.to_numeric(review["reported_line_number"], errors="coerce").astype("Int64")
findings["reported_line_number"] = pd.to_numeric(findings["reported_line_number"], errors="coerce").astype("Int64")
location_of = {place(r): r["location_key"] for r in findings.to_dict("records")}
review["why_review"] = review.apply(lambda r: "; ".join(review_reasons(r)), axis=1)
review["review_reasons"] = review.apply(lambda r: len(review_reasons(r)), axis=1)
SEV_RANK = {s: i for i, s in enumerate(SEVERITY_ORDER)}
review["_sev"] = review["severity_reported"].map(lambda s: min(SEV_RANK.get(x.strip(), 9) for x in str(s).split(";")))
review = review.sort_values(["review_reasons", "_sev"], ascending=[False, True]).drop(columns="_sev")
# One example per pilot and rule keeps the table varied; combined_manual_review.csv keeps every location.
examples = review[review["review_reasons"] > 0].drop_duplicates(["pilot", "rule_id"])[["pilot", "rule_id", "severity_reported", "file_path", "reported_line_number",
                                                  "evidence_snippet", "evidence_status", "configurations", "why_review"]].head(TOP_REVIEW_EXAMPLES)
print(f"{len(review)} distinct security locations; {int((review['review_reasons'] > 0).sum())} match at least one review rule.")
display(examples)

In [ ]:
# Reliability tables
outcomes = scans.pivot_table(index="pilot", columns="outcome", values="configuration", aggfunc="count", fill_value=0) \
    .reindex(index=list(RUNS), columns=[o for o in OUTCOME_ORDER if o in set(scans["outcome"])], fill_value=0)
failures = []
for r in scans.dropna(subset=["analyzers_failed"]).to_dict("records"):
    failures += [{"pilot": r["pilot"], "configuration": r["configuration"], "status": r["status"], "analyzer": f["analyzer"],
                  "failure_type": f["type"], "error": f["error"]} for f in json.loads(r["analyzers_failed"])]
failures = pd.DataFrame(failures, columns=["pilot", "configuration", "status", "analyzer", "failure_type", "error"])
evidence = findings.pivot_table(index="pilot", columns="evidence_status", values="finding_key", aggfunc="count", fill_value=0) \
    .reindex(index=list(RUNS), columns=["exact", "ambiguous", "mismatch", "unavailable"], fill_value=0) if not findings.empty else pd.DataFrame()
verdicts = []
for r in scans[scans["outcome"].isin(READABLE)].to_dict("records"):
    sec = security[(security["pilot"] == r["pilot"]) & (security["configuration"] == r["configuration"])]
    medium_or_lower = int(sec["severity"].isin(["MEDIUM", "LOW", "INFO"]).sum())
    if r["is_safe"] and len(sec):
        verdicts.append({"pilot": r["pilot"], "configuration": r["configuration"], "status": r["status"], "is_safe": r["is_safe"],
                         "max_severity": r["max_severity"], "security_findings": len(sec), "medium_or_lower": medium_or_lower})
verdicts = pd.DataFrame(verdicts)
print("Outcomes"); display(outcomes)
print("\nAnalyzer failures"); display(failures)
print("\nEvidence check"); display(evidence)
print("\nScans that Cisco calls is_safe=True although they contain security findings"); display(verdicts)

## 6 — Charts

Two charts, both from the primary group (complete scans): distinct security finding locations per pilot by
severity, and finding-level changes per comparison for the primary vs the sensitivity analysis.

In [ ]:
SEVERITY_COLORS = {"INFO": "#86b6ef", "LOW": "#5598e7", "MEDIUM": "#2a78d6", "HIGH": "#1c5cab", "CRITICAL": "#0d366b"}
FIGURES = []


def bar_chart(table, title, xlabel, colors, filename):
    """Horizontal grouped bars with counts at the bar ends; legend outside the plot."""
    if table.empty or int(table.to_numpy().sum()) == 0:
        print(f"(nothing to plot: {title})")
        return
    rows, series = list(table.index), list(table.columns)
    bar_h, peak = 0.8 / len(series), table.to_numpy().max()
    fig, ax = plt.subplots(figsize=(9, 1.0 + 0.4 * len(rows) * len(series)))
    for i, name in enumerate(series):
        y = [r - 0.4 + bar_h * (i + 0.5) for r in range(len(rows))]
        ax.barh(y, table[name], height=bar_h * 0.9, color=colors.get(name, "#777777"), label=str(name))
        for yy, v in zip(y, table[name]):
            if v:
                ax.text(v + peak * 0.01, yy, str(int(v)), va="center", fontsize=8, color="#444444")
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels([str(r) for r in rows], fontsize=9)
    ax.invert_yaxis()
    ax.set_xlim(0, peak * 1.12)
    ax.set_xlabel(xlabel)
    ax.set_title(title, loc="left", fontsize=11)
    ax.legend(fontsize=8, frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1.0))
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(axis="x", color="#dddddd", linewidth=0.6)
    ax.set_axisbelow(True)
    plt.tight_layout()
    fig.savefig(OUT_DIR / "figures" / filename, dpi=150, bbox_inches="tight")
    FIGURES.append((filename, title))
    plt.show()


bar_chart(sev_distinct_primary.loc[:, (sev_distinct_primary.sum() > 0)] if not sev_distinct_primary.empty else sev_distinct_primary,
          "Distinct security finding locations per pilot, by severity (complete scans)", "distinct locations",
          SEVERITY_COLORS, "severity_by_pilot.png")
change_table = pd.DataFrame({"primary: additional": totals_primary.set_index("comparison")["additional"],
                             "sensitivity: additional": totals_sensitivity.set_index("comparison")["additional"],
                             "primary: unchanged": totals_primary.set_index("comparison")["unchanged"],
                             "sensitivity: unchanged": totals_sensitivity.set_index("comparison")["unchanged"]})
bar_chart(change_table, "Finding changes per comparison: primary vs sensitivity analysis", "number of findings",
          {"primary: additional": "#eda100", "sensitivity: additional": "#eb6834",
           "primary: unchanged": "#2a78d6", "sensitivity: unchanged": "#1baf7a"}, "comparisons_primary_vs_sensitivity.png")

## 7 — Write the combined CSVs and `combined_report.md`

Every number in the report comes from the tables above. Limitation bullets are generated from the data.

In [ ]:
def md_cell(v):
    if v is None or (not isinstance(v, (str, list, dict)) and pd.isna(v)):
        return ""
    return str(v).replace("|", "\\|").replace("\n", " ")


def md_table(df, max_rows=25, index=True):
    if df is None or df.empty:
        return "_No data._\n"
    t = df.reset_index() if index and any(n is not None for n in df.index.names) else df
    head = [md_cell(c) for c in t.columns]
    lines = ["| " + " | ".join(head) + " |", "|" + "---|" * len(head)]
    lines += ["| " + " | ".join(md_cell(v) for v in row) + " |" for row in t.head(max_rows).itertuples(index=False)]
    return "\n".join(lines) + (f"\n\n_{len(t) - max_rows} more row(s) in the CSV files._" if len(t) > max_rows else "") + "\n"


def limitation_bullets():
    b = [f"Only {len(RUNS)} hand-picked pilot skill(s): these are illustrative observations, not estimates for GitSkills."]
    if not contract_scans.empty:
        b.append(f"{len(contract_scans)} scan(s) were CONTRACT_ONLY ({', '.join(contract_scans['pilot'] + ' / ' + contract_scans['configuration'])}); "
                 "comparisons that rely on them appear only in the sensitivity analysis.")
    zero_primary = totals_primary[totals_primary["pairs_used"] == 0]["comparison"].tolist()
    if zero_primary:
        b.append("No primary pair at all for: " + "; ".join(zero_primary) + ".")
    thin = totals_primary[(totals_primary["pairs_used"] > 0) & (totals_primary["pairs_used"] < 5)]
    if not thin.empty:
        b.append(f"Primary comparisons rest on {thin['pairs_used'].min()} to {thin['pairs_used'].max()} pair(s) each; differences between "
                 "configurations cannot be generalised.")
    if not evidence.empty and (evidence[["ambiguous", "mismatch"]].to_numpy().sum() > 0):
        b.append(f"{int(evidence[['ambiguous', 'mismatch']].to_numpy().sum())} finding record(s) have a reported line that does not hold their "
                 "evidence (Cisco's Markdown code-block line numbering); reported lines were kept unchanged.")
    if not evidence.empty and evidence["unavailable"].sum():
        b.append(f"{int(evidence['unavailable'].sum())} of {len(findings)} finding record(s) carry no snippet or no line, so their location "
                 "could not be checked automatically.")
    no_sec = [p for p in RUNS if not ((security["pilot"] == p) & (security["run_outcome"] == "complete")).any()]
    if no_sec:
        b.append(f"No security findings in complete scans for: {', '.join(no_sec)}. Absence of a finding is not evidence of absence of a "
                 "behavior; nothing here is a false-negative claim.")
    b += ["Cisco findings are rule matches, not confirmed vulnerabilities; the manual-review columns are intentionally blank.",
          "Finding records count once per configuration; distinct locations count each (skill, rule, file, line) once."]
    return b


R = [f"# Cisco Skill Scanner: combined pilot comparison\n",
     f"Generated by `notebooks/02_cisco_pilot_comparison.ipynb` on {datetime.now(timezone.utc):%Y-%m-%d %H:%M} UTC from {len(RUNS)} run folders. "
     f"Scanner `{SCANNER_VERSION}`, policy {POLICY}, configurations: {', '.join(CONFIG_ORDER)}. LLM and OSV analyzers were not used.\n",
     "> **Cisco findings are not confirmed vulnerabilities.** A finding means a Cisco rule matched an operation in a skill's files. "
     "Whether the operation is legitimate, risky or malicious requires manual review. **Primary** results use COMPLETE scans only; "
     "the **sensitivity analysis** adds CONTRACT_ONLY scans (only Cisco's known rule-registration error) and is reported separately. "
     "Contract-only scans are not assumed complete or accurate.\n",
     "## 1. Pilots and scans\n", md_table(overview, index=False),
     "\nFindings per configuration (all finding kinds; scan status in brackets):\n", md_table(per_config),
     "\nCisco's maximum severity per configuration:\n", md_table(max_sev),
     "\n## 2. Total findings and severity distributions (primary: complete scans)\n",
     "Security finding **records** by severity (one per finding per configuration):\n", md_table(sev_records_primary),
     "\nDistinct security finding **locations** by severity (each skill, rule, file, line once):\n", md_table(sev_distinct_primary),
     "\nSensitivity: distinct locations when contract-only scans are added:\n", md_table(sev_distinct_sensitivity),
     "\n## 3. SKILL.md only vs full folder, and default vs behavioral analyzer\n",
     "Sample sizes and excluded pairs per comparison (`excluded_*` = left out of the primary analysis):\n", md_table(sample_sizes()),
     "\n### Primary analysis (complete scans only)\n", md_table(totals_primary, index=False),
     "\nPer pilot:\n", md_table(per_pilot_changes(PRIMARY)),
     "\nFiles that contribute the additional findings when the full folder is scanned (primary; a SKILL.md row means Cisco re-read "
     "the instructions once the scripts were present):\n", md_table(files_new, index=False),
     "\n### Sensitivity analysis (complete + contract-only scans)\n", md_table(totals_sensitivity, index=False),
     "\nPer pilot:\n", md_table(per_pilot_changes(SENSITIVITY)),
     "\n## 4. COMPLETE vs CONTRACT_ONLY\n", "Contract-only scans:\n", md_table(contract_scans, index=False),
     "\nDoes including them change the comparisons?\n"]
R += [f"- {line}" for line in effect_lines]
if not invalid.empty:
    R.append(f"- {len(invalid)} finding record(s) in contract-only scans carry Cisco's own `rule_contract = invalid` flag "
             f"({', '.join(sorted(set(invalid['rule_id'])))}).")
if not rules_contract_only.empty:
    R += ["\nRules that appear **only** in contract-only scans (not in any complete scan):\n", md_table(rules_contract_only, index=False)]
R += ["\n## 5. Security categories and most frequent rules (primary, distinct locations)\n",
      "Cisco categories:\n", md_table(cat_primary),
      "\nResearch categories (our mapping; categories with no Cisco rule stay at zero):\n", md_table(research_primary),
      "\nRules:\n", md_table(rules_primary, index=False),
      "\n## 6. Examples that require manual security review\n",
      f"Selected by explicit rules (severity, context dependence, analyzer dependence, contract-only, unreliable location, manifest "
      f"mismatch), ranked by how many rules apply. {int((review['review_reasons'] > 0).sum())} of {len(review)} distinct security "
      f"locations match at least one rule; the table shows the highest-ranked example for each pilot and rule "
      f"({len(examples)} rows). `combined_manual_review.csv` lists every location. None has been reviewed yet.\n",
      md_table(examples, index=False),
      "\n## 7. Scanner reliability\n", "Scan outcomes:\n", md_table(outcomes),
      "\nAnalyzer failures reported by Cisco:\n", md_table(failures, index=False),
      "\nEvidence check of reported locations (finding records):\n", md_table(evidence),
      "\nScans that Cisco calls `is_safe = True` although they contain security findings:\n", md_table(verdicts, index=False)]
if FIGURES:
    R += ["\n## Figures\n"] + [f"![{t}](figures/{f})\n" for f, t in FIGURES]
R += ["\n## Limitations\n"] + [f"- {x}" for x in limitation_bullets()]
R += ["\n\n## Sources\n"] + [f"- `{p}`: `{RUN_DIRS[p].relative_to(ROOT)}`" for p in RUNS]
report_text = "\n".join(R) + "\n"

(OUT_DIR / "combined_report.md").write_text(report_text, encoding="utf-8")
overview.to_csv(OUT_DIR / "combined_overview.csv", index=False)
scans.to_csv(OUT_DIR / "combined_scan_summary.csv", index=False)
findings.to_csv(OUT_DIR / "combined_findings.csv", index=False)
pairs.to_csv(OUT_DIR / "combined_pairs.csv", index=False)
diffs.to_csv(OUT_DIR / "combined_comparison_findings.csv", index=False)
review.to_csv(OUT_DIR / "combined_manual_review.csv", index=False)
sha = lambda path: hashlib.sha256(path.read_bytes()).hexdigest()
(OUT_DIR / "combined_metadata.json").write_text(json.dumps({
    "written_at": datetime.now(timezone.utc).isoformat(timespec="seconds"), "scanner_version": SCANNER_VERSION, "policy": POLICY,
    "configurations": CONFIG_ORDER,
    "runs": {p: {"folder": str(RUN_DIRS[p].relative_to(ROOT)), "run_id": metadata[p].get("run_id"), "run_mode": metadata[p].get("run_mode"),
                 "files": {n: sha(RUN_DIRS[p] / n) for n in ("scan_summary.csv", "individual_findings.csv", "context_comparison.csv",
                                                              "analyzer_comparison.csv", "manual_review_template.csv") if (RUN_DIRS[p] / n).is_file()}}
             for p in RUNS}}, indent=2) + "\n", encoding="utf-8")
print("Saved to", OUT_DIR.relative_to(ROOT), ":", sorted(p.name for p in OUT_DIR.iterdir()))
display(Markdown(report_text.replace("](figures/", f"]({(OUT_DIR / 'figures').as_uri()}/")))

## Researcher report for this combined analysis

Builds `results/reports/<database>/combined/<this folder>/` with `START_HERE.md`, a self-contained `dashboard.html`
and the same four-folder layout as single runs (see Section 17 of notebook 01). The combined folder written above is
only read. Set `BUILD_REPORT = False` to skip.

In [ ]:
BUILD_REPORT = True

if BUILD_REPORT:
    import sys
    if str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))
    from gitskills_reports import build_report, load_combined, refresh_index

    REPORT_DIR = build_report(load_combined(OUT_DIR, ROOT), ROOT)
    refresh_index(ROOT)
    print("Report   :", REPORT_DIR.relative_to(ROOT))
    print("Dashboard:", (REPORT_DIR / "01_IMPORTANT_RESULTS" / "dashboard.html").as_uri())